# Macrocosm photo-z CNN — train your own version (Colab + MLflow)

**One self-contained notebook**, and it **streams the data — only one batch is ever in RAM**
(the image shards are memory-mapped on local disk, rows are read on the fly). So you can train on
the full 600k at 64×64 without a High-RAM runtime; RAM is no longer the limit — your local disk
(how many shards you download) is.

**To make your own version:** edit the **MODEL cell (section 2)**, name your run in the MLflow cell,
run all. **Runtime → Change runtime type → GPU** first.

## 0. Setup (Colab) — install, log in, pull data
Pull as many shards as you want to train on (each = 6000 cutouts, ~235 MB). They go to local disk;
streaming means downloading more costs **disk + download time, not RAM**.

In [ ]:
!pip -q install mlflow
from google.colab import auth; auth.authenticate_user()
!gcloud config set project macrocosm-lewagon -q
!mkdir -p /content/data
!gcloud storage cp gs://macrocosm-lewagon/data/sample_v1/catalog_v1.parquet /content/data/
for a in range(0, 24000, 6000):            # 4 shards = 24k cutouts (quick run)
    b = a + 6000
    !gcloud storage cp gs://macrocosm-lewagon/data/sample_v1/images_{a:07d}_{b:07d}.npy /content/data/
DATA_DIR = '/content/data'
# more data (streaming -> no RAM cost, just disk): raise the range, or pull all 91 shards:
#   !gcloud storage cp gs://macrocosm-lewagon/data/sample_v1/images_*.npy /content/data/

## 1. Pick the data — indices only (nothing big enters RAM)
We only build a list of row indices + their redshift labels here. The images are read later,
one batch at a time, straight from the memory-mapped shards.

In [ ]:
import glob, re, numpy as np, pandas as pd
SHARD = 6000
DATA_DIR = globals().get('DATA_DIR', '/content/data')

z_all = pd.read_parquet(f'{DATA_DIR}/catalog_v1.parquet', columns=['redshift'])['redshift'].values
n_shards = len(glob.glob(f'{DATA_DIR}/images_*.npy'))
n_avail  = min(len(z_all), n_shards * SHARD)          # rows whose images are downloaded
N = globals().get('N', n_avail)                       # default: use ALL downloaded rows
perm = np.random.RandomState(0).permutation(n_avail)[:N]
val_idx, train_idx = perm[:2000], perm[2000:]         # fixed 2k val
zva = z_all[val_idx]                                  # true redshift for val (callback + metric)
print(f'shards={n_shards}  available={n_avail}  using N={N}  ->  train {len(train_idx)} / val {len(val_idx)}')

## 2. 👉 THE MODEL — **edit this cell** to redesign the architecture
Example: VGG stem (cheap 64→16 downsample) + 3 lightweight **Inception** modules → GAP →
a 64-d **`embedding`** (feeds the fusion head later) → `z`. ~288K params. Change anything:
more/fewer Inception blocks, deeper stem, a classification head, transfer learning. (KB MCM-A-18)

In [ ]:
from tensorflow.keras import layers as L, Model, Input

def inception(x, f1, f3r, f3, f5r, f5, fp, name):
    """4 parallel branches concatenated on channels; 1x1 bottlenecks keep it cheap."""
    b1 = L.Conv2D(f1, 1, padding='same', activation='relu', name=f'{name}_1x1')(x)
    b3 = L.Conv2D(f3r, 1, padding='same', activation='relu', name=f'{name}_3x3_reduce')(x)
    b3 = L.Conv2D(f3, 3, padding='same', activation='relu', name=f'{name}_3x3')(b3)
    b5 = L.Conv2D(f5r, 1, padding='same', activation='relu', name=f'{name}_5x5_reduce')(x)
    b5 = L.Conv2D(f5, 5, padding='same', activation='relu', name=f'{name}_5x5')(b5)
    bp = L.MaxPool2D(3, strides=1, padding='same', name=f'{name}_pool')(x)
    bp = L.Conv2D(fp, 1, padding='same', activation='relu', name=f'{name}_pool_proj')(bp)
    return L.Concatenate(axis=-1, name=f'{name}_concat')([b1, b3, b5, bp])

def build_cnn(input_shape=(64, 64, 5), embed_dim=64):
    inp = Input(shape=input_shape, name='cutout')
    x = L.Conv2D(32, 3, padding='same', activation='relu', name='stem1a')(inp)
    x = L.Conv2D(32, 3, padding='same', activation='relu', name='stem1b')(x)
    x = L.BatchNormalization(name='stem1_bn')(x); x = L.MaxPool2D(name='stem1_pool')(x)   # 32x32
    x = L.Conv2D(64, 3, padding='same', activation='relu', name='stem2')(x)
    x = L.BatchNormalization(name='stem2_bn')(x); x = L.MaxPool2D(name='stem2_pool')(x)   # 16x16
    x = inception(x, 32, 32, 48, 8, 24, 24, name='inc1'); x = L.BatchNormalization(name='inc1_bn')(x)
    x = L.MaxPool2D(name='inc1_down')(x)                                                  # 8x8
    x = inception(x, 64, 48, 96, 16, 48, 48, name='inc2'); x = L.BatchNormalization(name='inc2_bn')(x)
    x = L.MaxPool2D(name='inc2_down')(x)                                                  # 4x4
    x = inception(x, 64, 48, 96, 16, 48, 48, name='inc3'); x = L.BatchNormalization(name='inc3_bn')(x)
    x = L.GlobalAveragePooling2D(name='gap')(x)
    x = L.Dense(128, activation='relu', name='dense')(x); x = L.Dropout(0.3, name='dropout')(x)
    emb = L.Dense(embed_dim, activation='relu', name='embedding')(x)
    zout = L.Dense(1, name='z')(emb)
    return Model(inp, zout, name='photoz_cnn')

def build_embedder(cnn):
    return Model(cnn.input, cnn.get_layer('embedding').output, name='photoz_embedder')

model = build_cnn()
model.summary()
print('params:', f'{model.count_params():,}')

## 3. Streaming pipeline + metrics — usually leave as-is
`streaming_dataset` memory-maps the shards and yields cropped rows on the fly, so **only the
current batch is in RAM**. Same arcsinh + per-image norm as the serving backend (no skew).

In [ ]:
import tensorflow as tf

def preprocess(img, label):
    img = tf.cast(img, tf.float32); img = tf.math.asinh(img)
    m = tf.reduce_mean(img, axis=[0, 1], keepdims=True)
    s = tf.math.reduce_std(img, axis=[0, 1], keepdims=True) + 1e-6
    return (img - m) / s, label

def augment(img, label):
    img = tf.image.rot90(img, tf.random.uniform([], 0, 4, dtype=tf.int32))
    img = tf.image.random_flip_left_right(img); img = tf.image.random_flip_up_down(img)
    return img, label

def streaming_dataset(idx, training=False, batch=128, crop=64):
    """Only batches enter RAM: mmap the shards (lazy), read+crop rows on the fly.
    Training reshuffles every epoch (gen() re-runs); val keeps idx order (for the metric)."""
    paths = sorted(glob.glob(f'{DATA_DIR}/images_*.npy'),
                   key=lambda p: int(re.findall(r'images_(\d+)_', p)[0]))
    mm = [np.load(p, mmap_mode='r') for p in paths]      # lazy memmaps, NOT in RAM
    o = (64 - crop) // 2; idx = np.asarray(idx)
    def gen():
        order = np.random.permutation(idx) if training else idx   # fresh shuffle each epoch
        for i in order:
            s, r = divmod(int(i), SHARD)
            yield np.asarray(mm[s][r][o:o+crop, o:o+crop, :], 'float32'), np.float32(np.log1p(z_all[i]))
    sig = (tf.TensorSpec((crop, crop, 5), tf.float32), tf.TensorSpec((), tf.float32))
    ds = tf.data.Dataset.from_generator(gen, output_signature=sig)
    ds = ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE).repeat()   # re-runs gen() -> reshuffles
    return ds.batch(batch).prefetch(tf.data.AUTOTUNE)    # prefetch overlaps disk IO with the GPU

def _dz(zt, zp): zt, zp = np.asarray(zt), np.asarray(zp); return (zp - zt) / (1 + zt)
def sigma_mad(zt, zp): d = _dz(zt, zp); return float(1.4826 * np.median(np.abs(d - np.median(d))))
def outlier_rate(zt, zp): return float(np.mean(np.abs(_dz(zt, zp)) > 0.05))

class SigmaMadCallback(tf.keras.callbacks.Callback):
    def __init__(self, val_ds, z_true): super().__init__(); self.val_ds = val_ds; self.z_true = np.asarray(z_true)
    def on_epoch_end(self, epoch, logs=None):
        zp = np.expm1(self.model.predict(self.val_ds, verbose=0).ravel())
        sm, out = sigma_mad(self.z_true, zp), outlier_rate(self.z_true, zp)
        logs = logs if logs is not None else {}
        logs['val_sigma_MAD'] = sm; logs['val_outlier'] = out
        print(f'  -> val sigma_MAD={sm:.4f}  outlier={out*100:.1f}%')

## 4. Connect MLflow (optional)
Paste the bearer token (ask the team — not in git); start the server first (`make mlflow-start`).
Without a token it just trains and skips logging.

In [ ]:
import os
MLFLOW_TOKEN = 'PASTE_MLFLOW_API_TOKEN_HERE'
USE_MLFLOW = 'PASTE' not in MLFLOW_TOKEN
if USE_MLFLOW:
    import mlflow, mlflow.tensorflow
    os.environ['MLFLOW_TRACKING_URI'] = 'https://146-148-10-86.sslip.io'
    os.environ['MLFLOW_TRACKING_TOKEN'] = MLFLOW_TOKEN
    mlflow.set_experiment('photoz-cnn')
    mlflow.tensorflow.autolog(log_models=True)
    print('MLflow: logging to', os.environ['MLFLOW_TRACKING_URI'])
else:
    print('MLflow token not set -> training without logging (fine for a quick test)')

## 5. Train + log
Builds the **streaming** datasets (one batch in RAM) and trains. Name **your** run. Logs the full
**recipe** to MLflow — a `CONFIG` of preprocessing+training params *and* a `recipe.py` artifact with
the exact source of `preprocess` / `augment` / `streaming_dataset` / `build_cnn` — so every run is
reproducible and comparable (you can see exactly what preprocessing each run used).

In [ ]:
EPOCHS = globals().get('EPOCHS', 50)
RUN_NAME = 'example_vgg_inception'      # <- name YOUR run

import inspect
CROP, BATCH, LR = 64, 128, 1e-3            # config vars -> logged (not hard-coded in the log)
train_ds = streaming_dataset(train_idx, training=True, batch=BATCH, crop=CROP)
val_ds   = streaming_dataset(val_idx,   training=False, batch=256, crop=CROP)
steps = len(train_idx) // BATCH                     # repeated train stream -> set the cadence
val_steps = int(np.ceil(len(val_idx) / 256))        # finite val stream -> set its cadence too
model.compile(optimizer=tf.keras.optimizers.Adam(LR),
              loss=tf.keras.losses.Huber(delta=0.02), metrics=['mae'])
cbs = [SigmaMadCallback(val_ds, zva),
       tf.keras.callbacks.EarlyStopping('val_loss', patience=6, restore_best_weights=True),
       tf.keras.callbacks.ReduceLROnPlateau('val_loss', factor=0.5, patience=3, min_lr=1e-5)]

# everything that defines this run -> logged to MLflow (table) + recipe.py artifact (exact code)
CONFIG = dict(crop=CROP, batch=BATCH, lr=LR, optimizer='adam', loss='huber(0.02)', target='log1p(z)',
              stretch='arcsinh', norm='per-image per-channel', augment='rot90+flip',
              arch='vgg-stem+3inception', N=len(train_idx), params=int(model.count_params()))

def run_training():
    model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, steps_per_epoch=steps, validation_steps=val_steps, callbacks=cbs)
    zp = np.expm1(model.predict(val_ds, verbose=0).ravel())
    print('val sigma_MAD =', round(sigma_mad(zva, zp), 5), ' (tabular baseline 0.0133)')
    return zp

if USE_MLFLOW:
    import mlflow
    with mlflow.start_run(run_name=RUN_NAME):
        mlflow.log_params(CONFIG)                                  # preprocessing + training config
        mlflow.log_text('\n\n'.join(inspect.getsource(f)         # the exact preprocessing/model code
                        for f in (preprocess, augment, streaming_dataset, build_cnn)), 'recipe.py')
        zp = run_training()
        mlflow.log_metrics({'final_val_sigma_MAD': sigma_mad(zva, zp),
                            'final_val_outlier': outlier_rate(zva, zp)})
else:
    run_training()

## 6. Compare
Open the MLflow UI (`https://146-148-10-86.sslip.io`, GitHub-org login) and compare your run vs
the example + teammates'. Streaming means you can scale to **64px × 600k** (download all 91 shards)
without RAM trouble. Beat 0.0133 (tabular) with images, or push toward Pasquet's 0.0091.